# Notebook A — Dynamique de croissance bactérienne

**Notebook A — Stage de lycée · Laboratoire de microbiologie quantitative  (version stagiaire)**

Ce notebook explore comment des règles simples au niveau de la cellule individuelle
produisent les courbes de croissance observées en labo.

---

### Comment utiliser ce notebook
- Exécuter chaque cellule dans l'ordre : **Shift + Entrée**
- Lire les questions encadrées `>` et y réfléchir avant de passer à la suite
- Modifier les paramètres et observer ce qui change

> **Environnement recommandé** : [Google Colab](https://colab.research.google.com) — aucune installation nécessaire.
> En local : `pip install numpy matplotlib pandas`
>
> **Durée estimée** : 4 demi-journées (sections A1–A4)

## Imports

In [ ]:
# Importer numpy sous le nom np
# Importer matplotlib.pyplot sous le nom plt

# Configurer la résolution : plt.rcParams['figure.dpi'] = 100
# Configurer la police     : plt.rcParams['font.size']  = 12
# Afficher "Bibliotheques chargees." 

---
## Section A1 — Croissance exponentielle : le modèle déterministe

Une bactérie se divise en deux. Ses deux filles se divisent en deux. Et ainsi de suite.

Si à chaque pas de temps, chaque cellule a une probabilité $p$ de se diviser,
le nombre moyen de nouvelles divisions est $p \times N$.

**Modèle déterministe** : on ignore les fluctuations et on écrit directement :

$$N(t+1) = N(t) \times (1 + p)$$

C'est une **multiplication répétée par le même facteur** $(1+p)$.

> **Avant de coder** : si $N_0 = 10$ et $p = 0.1$, quel est $N$ après 1 pas ?
> Après 2 pas ? Calculez à la main les 3 premières valeurs.

In [ ]:
# ── Définir les paramètres ─────────────────────────────────────────────────────
# N0=10, p=0.1, n_steps=60

# ── Simulation déterministe ─────────────────────────────────────────────────────
# N = N0  ;  Ns_det = [N0]
# for i in range(n_steps) :
#     N = N * (1 + p)    # multiplication répétée
#     Ns_det.append(N)

# ── Tracé linéaire ─────────────────────────────────────────────────────────────
# plt.figure(figsize=(8, 4))
# Tracer Ns_det  (color='crimson', lw=2)
# Labels, titre, tight_layout, show

### Représentation logarithmique

En **échelle logarithmique**, la multiplication répétée par $(1+p)$ se transforme
en une **droite** — ce qui permet d'identifier visuellement la phase de croissance
exponentielle sur vos courbes OD.

Plus la pente est forte, plus les bactéries se divisent vite.

> **Question** : quel est le lien entre la pente de cette droite et le paramètre $p$ ?
> Et avec le **temps de doublement** des bactéries ?

In [ ]:
# Reprendre Ns_det de la cellule précédente
# Ajouter plt.yscale('log') avant plt.show()
# Labels, titre, tight_layout, show

# Note : pour le temps de doublement, lisez directement sur le graphe
# à quel pas N a doublé par rapport à N0.
# (np.log = logarithme, vu en terminale)

---
## Section A2 — Simulation stochastique : le hasard cellule par cellule

Le modèle déterministe donne la **valeur moyenne attendue**.
Mais en réalité, chaque cellule décide de diviser ou non de façon **aléatoire**.

**Règle microscopique** : à chaque pas de temps, pour *chaque* cellule individuellement,
on tire au sort — elle divise avec probabilité $p$, indépendamment de ses voisines.

> **Question** : si le hasard joue un rôle à chaque cellule,
> pourquoi obtient-on une courbe régulière au niveau de la population ?

On va simuler cela directement et observer ce qui se passe.

---

> ⚠️ **Note technique** : cette simulation suppose $p \ll 1$ (probabilité petite par pas de temps).
> C'est l'approximation dite de *τ-leaping*, version discrète de l'algorithme de Gillespie.
> Pour $p = 0.1$, l'approximation est très bonne.

In [ ]:
# Définir la fonction simulate_one(N0, p, n_steps) qui retourne Ns :
#   N = N0 ; Ns = [N0]
#   for _ in range(n_steps) :
#       divisions = np.sum(np.random.random(N) < p)
#       N += divisions ; Ns.append(N)
#   return Ns

# Appeler simulate_one(N0=10, p=0.1, n_steps=60) → Ns_stoch
# Tracer Ns_det (rouge, lw=2.5) et Ns_stoch (bleu, alpha=0.8) sur même figure
# Labels, légende, titre, tight_layout, show

In [ ]:
# n_traj = 30  ;  all_traj = []
# fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Boucle sur n_traj :
#   traj = simulate_one(N0=10, p=0.1, n_steps=60)
#   all_traj.append(traj)
#   Tracer traj sur les 2 axes (alpha=0.15, steelblue, lw=1)
# mean_traj = np.mean(all_traj, axis=0)
# Sur chaque axe : Ns_det (rouge, lw=2.5) + mean_traj (noir pointillé, lw=2)
# axes[1].set_yscale('log')
# Titres, labels, légendes, suptitle, tight_layout, show

> **Observation** : chaque trajectoire est différente, mais leur **moyenne** suit
> de près la courbe déterministe. En échelle log, les trajectoires restent
> groupées autour d'une droite.
>
> C'est la **loi des grands nombres** : des règles aléatoires au niveau individuel
> produisent un comportement prévisible au niveau de la population.
>
> **À explorer** : relancez la simulation avec $N_0 = 2$ au lieu de $N_0 = 10$.
> Les trajectoires sont-elles plus ou moins dispersées ? Pourquoi ?

---
## Section A3 — Épuisement des nutriments et phase stationnaire

Dans un milieu fermé (tube, microplaque), les nutriments s'épuisent au fur et à mesure
que les bactéries se multiplient. La croissance finit par s'arrêter.

**Idée** : la probabilité de division $p$ n'est plus constante —
elle dépend de la concentration en nutriments $S$ disponibles.

### La courbe de Monod

$$p(S) = p_{\max} \times \frac{S}{K_S + S}$$

| Situation | Valeur de $p(S)$ | Interprétation |
|-----------|:---------------:|----------------|
| $S \gg K_S$ | $\approx p_{\max}$ | Nutriments abondants → croissance maximale |
| $S = K_S$ | $p_{\max}/2$ | $K_S$ fixe l'échelle de saturation |
| $S \ll K_S$ | $\approx 0$ | Nutriments épuisés → croissance arrêtée |

Cette forme est identique à la cinétique enzymatique de Michaelis-Menten.

In [ ]:
# p_max = 0.1  ;  Ks = 20.0
# S_vals = np.linspace(0, 200, 400)
# p_vals = p_max * S_vals / (Ks + S_vals)
# plt.figure(figsize=(7, 4))
# Tracer p_vals vs S_vals (darkgreen, lw=2.5)
# axhline à p_max (gris tirets) ; axhline à p_max/2 (gris pointillés)
# axvline à Ks (rouge tirets)
# Labels, titre, légende, tight_layout, show

In [ ]:
# Définir simulate_nutrients(N0, S0, p_max, Ks, Y, n_steps) :
#   N, S = float(N0), float(S0)  ;  Ns, Ss = [N], [S]
#   for _ in range(n_steps) :
#       p = p_max * S / (Ks + S)
#       divisions = np.sum(np.random.random(int(N)) < p)
#       N += divisions  ;  S -= divisions / Y  ;  S = max(S, 0.0)
#       Ns.append(N)  ;  Ss.append(S)
#   return Ns, Ss

# Paramètres : N0_m=10, S0_m=500, p_max_m=0.1, Ks_m=20.0, Y_m=5.0, n_m=200
# fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
# Boucle 15 trajectoires :
#   Ns_m, Ss_m = simulate_nutrients(...)
#   axes[0].plot(Ns_m, alpha=0.2, steelblue)
#   axes[1].plot(Ss_m, alpha=0.2, darkorange)
# axes[0].set_yscale('log')  ;  Labels, titre, tight_layout, show

### Prédiction de la capacité portante

Dans notre modèle, chaque division consomme exactement $1/Y$ unités de nutriments
($Y$ est le **rendement** : nombre de cellules produites par unité de nutriment).

Si on part de $S_0$ nutriments et $N_0$ cellules, quand tous les nutriments sont épuisés :

$$N_{\max} = N_0 + Y \times S_0$$

**Ce n'est pas un paramètre ajusté — c'est une prédiction du modèle.**
La cellule suivante va vérifier cette prédiction numériquement.

In [ ]:
# N_max_pred = N0_m + Y_m * S0_m  → afficher avec print()

# Vérification : lancer 50 simulations, récupérer la valeur finale [0][-1]
# N_finals = [simulate_nutrients(...)[0][-1] for _ in range(50)]
# Afficher np.mean(N_finals) et np.std(N_finals)
# Comparer à N_max_pred

> **Note** : la simulation donne typiquement ~1 % de plus que la prédiction
> $N_{\max} = N_0 + Y \cdot S_0$. La prédiction suppose une coupure nette
> quand $S = 0$, mais dans le modèle discret $p(S) \to 0$ progressivement —
> quelques divisions ont encore lieu quand $S$ est très petit.
> C'est un écart attendu, pas un bug.

---
## Section A4 — Comparaison avec les données expérimentales

### Premier test : la biomasse produite est-elle proportionnelle au glucose ?

Le modèle de Monod prédit $N_{\max} = N_0 + Y \times S_0$,
soit une relation **linéaire** entre biomasse produite et glucose initial,
avec une droite qui passe par l'origine.

On peut le vérifier directement avec vos données de mardi :
pour chaque tube, on calcule $\Delta\text{OD} = \text{OD}_{\max} - \text{OD}_0$.
Si le modèle est juste, les trois points doivent être alignés sur une droite passant par zéro.

Le **rendement** $Y$ est la pente de cette droite :
combien d'unités OD de bactéries sont produites par gramme de glucose consommé.

In [ ]:
# ── Entrez vos valeurs mesurées mardi ──────────────────────────────────────────
# glucose_gL    = [0.2, 2.0, 4.0]
# OD_initial    = 0.005
# OD_max_mesure = [???, ???, ???]   # à remplir avec vos valeurs

# ── Biomasse produite ──────────────────────────────────────────────────────────
# delta_OD = [od - OD_initial for od in OD_max_mesure]
# Afficher delta_OD pour chaque concentration

# ── Rendement Y : moyenne des pentes individuelles ─────────────────────────────
# Y_vals = [dod / glu for dod, glu in zip(delta_OD, glucose_gL)]
# Y_fit  = sum(Y_vals) / len(Y_vals)

# ── Tracé ΔOD = f([glucose]) ───────────────────────────────────────────────────
# glu_dense = np.linspace(0, max(glucose_gL) * 1.1, 100)
# plt.figure(figsize=(7, 4))
# plt.plot(glucose_gL, delta_OD, 'ko', ms=10, zorder=4, label='Données (mardi)')
# plt.plot(glu_dense, [Y_fit * g for g in glu_dense], 'b--', label=f'y = Y·x')
# Labels, légende, titre, tight_layout, show
# Afficher Y_fit et Y_fit * 2e9 (conversion en cellules/g/L)

### Lecture des données du plate reader
Le plate reader a mesuré l'OD600 de vos cultures toutes les quelques minutes
pendant une nuit entière. On va charger ces données pour
1.  vérifier la biomasse produite en fonction des nutriments pour differentes concentrations de glucose et de lactose.
2. mesurer le taux de croissance dans ces differentes conditions
3. visualiser les courbes de croissance en diauxie.

#### Format du fichier BioTek Gen5

Même format que dans le notebook d'induction : deux blocs TSV (OD + GFP),
avec pour chaque bloc le nom du canal en première ligne.
La fonction `parse_biotek` ci-dessous gère ce format.

> **Note** : si vous avez mesuré des courbes de croissance manuellement (OD au spectro),
> vous pouvez charger un CSV simple avec `pd.read_csv()` :
> colonne `temps_h` (heures) + une colonne par tube.

In [56]:
import pandas as pd

def parse_biotek(filepath):
    # Lit un fichier plate reader BioTek Gen5 (format TSV multi-blocs).
    # Retourne un dict {nom_canal : DataFrame}
    # Chaque DataFrame : colonne 'temps_h' (heures decimales) + colonnes A1...H12.
    with open(filepath, encoding='utf-8', errors='replace') as f:
        lines = f.read().splitlines()
    block_starts = []
    for i, line in enumerate(lines):
        s = line.strip()
        if s and chr(9) not in s and not s[0].isdigit():
            block_starts.append(i)
    result = {}
    for b_idx, b_start in enumerate(block_starts):
        canal_name = lines[b_start].strip()
        if b_start + 2 >= len(lines):
            continue
        header = lines[b_start + 2].strip().split(chr(9))
        b_end = block_starts[b_idx+1] if b_idx+1 < len(block_starts) else len(lines)
        rows = []
        for line in lines[b_start + 3 : b_end]:
            parts = line.strip().split(chr(9))
            if len(parts) < 3:
                continue
            t_str = parts[0]
            if not t_str or t_str == '0:00:00':
                continue
            t_parts = t_str.split(':')
            if len(t_parts) != 3:
                continue
            try:
                h, m, s = int(t_parts[0]), int(t_parts[1]), int(t_parts[2])
            except ValueError:
                continue
            rows.append([h + m/60 + s/3600] + parts[2:])
        if not rows:
            continue
        puits = header[2:]
        n_cols = len(rows[0]) - 1
        cols = ['temps_h'] + puits[:n_cols]
        df = pd.DataFrame(rows, columns=cols)
        for col in df.columns[1:]:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        result[canal_name] = df
    return result


# ══════════════════════════════════════════════════════════════════════════════════
# Pour charger VOS données BioTek, décommentez :
# ══════════════════════════════════════════════════════════════════════════════════
# data  = parse_biotek('votre_diauxie.txt')
# od_df = next(df for name, df in data.items() if '600' in name)
#
# Pour un CSV manuel (temps + une colonne par tube) :
# od_df = pd.read_csv('croissance_manuelle.csv').rename(columns={'temps': 'temps_h'})

# ── Données synthétiques diauxie — À REMPLACER par vos données réelles ─────────
np.random.seed(3)

def generer_diauxie():
    # Simule les courbes BioTek d'une expérience diauxie overnight.
    # Puits A1-A3 : glucose seul (3 concentrations)
    # Puits B1-B2 : glucose + lactose (diauxie)
    # Puits H12   : milieu seul (blanc)
    n_pts = 72
    temps = np.linspace(0, 14, n_pts)

    def logistic(t, K, t_mid, mu):
        return K / (1 + np.exp(-mu * (t - t_mid)))

    od = {'temps_h': temps}
    # Glucose seul : 3 concentrations → 3 capacités portantes différentes
    for label, K in [('A1', 0.35), ('A2', 0.80), ('A3', 1.40)]:
        vals = 0.05 + logistic(temps, K, K*3.8, 1.8)
        od[label] = np.clip(vals + np.random.normal(0, 0.008, n_pts), 0, None)
    # Glucose + Lactose : diauxie (deux phases avec creux intermédiaire)
    for label in ['B1', 'B2']:
        phase1 = logistic(temps, 0.45, 2.5, 2.2)
        # Phase lactose démarre après induction lacZ (~5h de lag total)
        phase2 = np.array([logistic(t, 0.42, 9.0, 1.6) if t > 5.0 else 0.0
                           for t in temps])
        vals = 0.05 + phase1 + phase2
        od[label] = np.clip(vals + np.random.normal(0, 0.010, n_pts), 0, None)
    # Blanc
    od['H12'] = np.clip(0.082 + np.random.normal(0, 0.003, n_pts), 0, None)

    return pd.DataFrame(od)

od_df = generer_diauxie()
print("Donnees chargees.")
print(f"  {len(od_df)} points, {len(od_df.columns)-1} puits")
print(f"  Duree : {od_df['temps_h'].min():.1f} – {od_df['temps_h'].max():.1f} h")
print(od_df[['temps_h', 'A1', 'A2', 'B1', 'H12']].head(3).to_string(index=False))

Donnees chargees.
  72 points, 6 puits
  Duree : 0.0 – 14.0 h
 temps_h       A1       A2       B1      H12
0.000000 0.093580 0.052112 0.062726 0.081981
0.197183 0.093799 0.068923 0.053059 0.086503
0.394366 0.105564 0.060650 0.058268 0.079390


### Relation biomasse / nutriments — vérification sur le plate reader

Le plate reader nous donne l'OD600 de **beaucoup** de conditions simultanément.
On va d'abord tracer toutes les courbes de croissance, extraire l'OD maximale
de chaque puits, puis vérifier que $\Delta\text{OD} \propto [\text{glucose}]$.

On pourra aussi comparer le rendement $Y$ mesuré ici avec celui obtenu mardi
à la main — et l'étendre aux puits **lactose seul** si le layout le permet.

In [ ]:
# ── Layout (ADAPTER) ───────────────────────────────────────────────────────────
# puits_glu = ['A1','A2','A3']  ;  puits_lac = []  ;  blanc = 'H12'
# glu_gL = [0.2, 2.0, 4.0]
# od_blanc = od_df[blanc].mean() if blanc in od_df.columns else 0.0

# ── Tracé courbes de croissance ────────────────────────────────────────────────
# plt.figure(figsize=(9, 4))
# Boucle for puit in puits_glu + puits_lac :
#   if puit in od_df.columns :
#       plt.plot(od_df['temps_h'], od_df[puit] - od_blanc, lw=1.5, label=puit)
# Labels, légende, titre, tight_layout, show

# ── ΔOD et rendement Y (plate reader) ─────────────────────────────────────────
# def delta_od_puit(puit) :
#   od_corr = od_df[puit] - od_blanc
#   return od_corr.max() - od_corr.iloc[0]
# delta_glu_pr = [delta_od_puit(p) for p in puits_glu if p in od_df.columns]
# Y_vals_glu_pr = [dod/glu for dod,glu in zip(delta_glu_pr, glu_gL)]
# Y_fit_pr = sum(Y_vals_glu_pr) / len(Y_vals_glu_pr)
# Tracer ΔOD = f([glucose]) + droite y=Y_fit_pr*x
# Afficher Y_fit_pr et comparer à Y_fit (mardi)

### Comparaison avec le modèle de Monod

Maintenant que les données sont chargées, comparons directement la forme
d'une courbe **glucose seul** avec la simulation de Monod.
Si le modèle est cohérent, les deux courbes normalisées doivent se superposer :
même transition exponentielle → stationnaire.

In [ ]:
# ── Mesurer µ : pente de log(OD) vs temps en phase exponentielle ───────────────
# OD_min_exp = 0.02  ;  OD_max_exp = 0.30  (adapter si besoin)
# fig, axes = plt.subplots(1, 2, figsize=(13, 4))
# Boucle for puit in puits_glu :
#   od_corr = od_df[puit] - od_blanc
#   mask = (od_corr > OD_min_exp) & (od_corr < OD_max_exp)
#   Si mask.sum() < 4 → continuer (pas assez de points)
#   t_exp = od_df['temps_h'][mask].values  ;  od_exp = od_corr[mask].values
#   coeffs = np.polyfit(t_exp, np.log(od_exp), 1)   # np.log = logarithme (terminale)
#   mu = coeffs[0]
#   axes[0].plot(od_df['temps_h'], od_corr, lw=1.5, label=puit)
#   od_fit = np.exp(coeffs[1] + coeffs[0]*t_exp)   # np.exp = inverse du log (terminale)
#   axes[0].plot(t_exp, od_fit, '--', color='gray', lw=1.2, alpha=0.8)
#   od_pos = od_corr[od_corr>0.005]
#   axes[1].plot(od_df['temps_h'][od_corr>0.005], np.log(od_pos), lw=1.5, label=puit)
# Labels, titres, légendes pour les 2 axes  ;  plt.tight_layout()  ;  plt.show()
# Afficher µ et temps de doublement : 0.693/mu*60 min

### Taux de croissance $\mu$

En phase exponentielle, l'OD double à intervalles réguliers.
La vitesse de doublement — le **taux de croissance** $\mu$ (en h⁻¹) — dépend-elle
de la concentration en glucose ?

En échelle logarithmique, la phase exponentielle apparaît comme une **droite**,
dont la pente est exactement $\mu$.

> **Note** : cette mesure utilise la fonction logarithme (`np.log`),
> étudiée en terminale. Retenez simplement : *pente plus forte = croissance plus rapide*.

In [ ]:
# ── Layout diauxie (ADAPTER) ────────────────────────────────────────────────────
# puits_diauxie = ['B1', 'B2']
# fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Gauche : linéaire ──────────────────────────────────────────────────────────
# Boucle sur puits_glu  : axes[0].plot(..., '--', lw=1.2, alpha=0.7, label=f'{puit} (glu)')
# Boucle sur puits_diauxie : axes[0].plot(..., '-', lw=2, label=f'{puit} (glu+lac)')
# Labels, titre, légende pour axes[0]

# ── Droite : log ───────────────────────────────────────────────────────────────
# Boucle sur puits_diauxie :
#   od_corr = od_df[puit] - od_blanc
#   mask = od_corr > 0.01
#   axes[1].plot(od_df['temps_h'][mask], np.log(od_corr[mask]), '-', lw=2)
# Labels, titre pour axes[1]
# plt.tight_layout()  ;  plt.show()

### La diauxie : croissance sur un mélange de nutriments

On va superposer les courbes de croissance **glucose seul** et **glucose + lactose**
pour identifier la pause caractéristique de la diauxie.

#### Diauxie — rappel

Quand *E. coli* croît sur glucose + lactose, elle consomme d'abord le glucose
(source préférée), puis marque une **pause** le temps d'induire les enzymes du
lactose (lacZ, lacY), avant de repartir sur le lactose. Cette double croissance
donne son nom à la *diauxie* (du grec *deux fois*) — et explique pourquoi
l'opéron *lac* est régulé !

> **À observer** : la hauteur du premier plateau dépend-elle de [glucose] ?
> La hauteur finale dépend-elle de [lactose] ?

In [ ]:
# ── Choisir une courbe de référence (glucose seul) ─────────────────────────────
# col_ref = puits_glu[1] si disponible, sinon puits_glu[0]
# OD_ref  = od_df[col_ref] - od_blanc

# ── Normaliser entre 0 et 1 ────────────────────────────────────────────────────
# OD_norm = (OD_ref - OD_ref.min()) / (OD_ref.max() - OD_ref.min())

# ── Simulation Monod normalisée ────────────────────────────────────────────────
# Ns_c, _ = simulate_nutrients(N0_m, S0_m, p_max_m, Ks_m, Y_m, n_m)
# Ns_arr  = np.array(Ns_c, dtype=float)
# Ns_norm = (Ns_arr - Ns_arr.min()) / (Ns_arr.max() - Ns_arr.min())
# t_sim   = np.linspace(od_df['temps_h'].min(), od_df['temps_h'].max(), len(Ns_norm))

# ── Tracé ─────────────────────────────────────────────────────────────────────
# plt.figure(figsize=(9, 5))
# Tracer OD_norm  ('ko-', ms=5, label=f'Données ({col_ref})')
# Tracer Ns_norm  ('b-',  lw=2, label='Simulation Monod')
# Labels, titre, légende, tight_layout, show

---
## Pour aller plus loin

1. **Modifier les paramètres** : que se passe-t-il biologiquement si `p_max` augmente ?
   Si `Ks` diminue ? Lequel des deux est plus facile à modifier expérimentalement ?

2. **Deux souches** : simuler deux populations avec des `p_max` différents partageant
   le même milieu (même réservoir $S$). Laquelle domine en fin de culture ? Pourquoi ?

3. **Temps de doublement** : à partir de vos données expérimentales, estimez le
   temps de doublement en phase exponentielle.
   *Indice* : c'est la pente de la droite en échelle log.

4. **Variabilité** : comparez la dispersion des trajectoires entre la phase
   exponentielle (N petit) et la phase stationnaire (N grand). Pourquoi change-t-elle ?